# IndieFake (Indian-accent) SSL training pipeline -- Colab

Picks up the `ai_voice_detector` pipeline at step 4 (SSL feature extraction), which was too slow on an 8GB CPU-only machine. Uses Colab's GPU (`Runtime > Change runtime type > T4 GPU`) instead.

**Before running**: put the 4 IndieFake zip parts (`drive-download-...-00{1..4}.zip`) somewhere in your Google Drive and set `INDIEFAKE_ZIP_DIR` below to that folder.

Steps: mount Drive -> clone repo -> re-fetch ASVspoof + real-world audio (public/deterministic sources, not stored in git) -> run the Indian dataset pipeline (organize -> verify -> degrade -> manifests -> extract embeddings on GPU) -> train -> evaluate (5-quadrant + leave-one-generator-out) -> push results back.

In [28]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
# EDIT THIS to wherever the 4 IndieFake zip parts live in your Drive
INDIEFAKE_ZIP_DIR = "/content/drive/MyDrive/IndieFake  Dataset"

In [4]:
REPO_URL = "https://github.com/Jeevan-Cyber-Sai/sih.git"
!git clone $REPO_URL /content/sih
%cd /content/sih/ai_voice_detector

Cloning into '/content/sih'...
remote: Enumerating objects: 6685, done.
remote: Total 6685 (delta 0), reused 0 (delta 0), pack-reused 6685 (from 1)
Receiving objects: 100% (6685/6685), 783.03 MiB | 33.69 MiB/s, done.
Resolving deltas: 100% (50/50), done.
Updating files: 100% (11681/11681), done.
/content/sih/ai_voice_detector


In [5]:
!pip install -q -r requirements.txt

In [6]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- set Runtime > Change runtime type > T4 GPU')

CUDA available: True
device: Tesla T4


## Re-fetch ASVspoof + real-world audio

`data/`, `data_realworld/` audio isn't in git (gitignored, large binary data) -- only their SSL embeddings are (`cache/ssl_embeddings/`, portable across machines since the cache key is now a relative path, not an absolute one). These scripts rebuild the audio deterministically (same seeds, same public sources) so file *names* line up with what's already cached; `LA_D_*`/WhatsApp/test-dir files come from the same seeded/tracked sources so they're byte-identical, while gTTS/LibriSpeech content could differ by a negligible amount from re-synthesis -- acceptable since only filenames need to match for cache lookups, and any recompute needed for a mismatch is cheap on GPU anyway.

In [7]:
!python scripts/download_asvspoof_subset.py 700

[18:45:30] opening remote LA.zip (range requests)...
[18:45:38] remote zip length=7640952520, entries=122328
[18:45:38] dev protocol parsed: bonafide=2548 spoof=22296
[18:45:52] progress 25/1400  ok=25 skip=0 fail=0
[18:46:04] progress 50/1400  ok=50 skip=0 fail=0
[18:46:14] progress 75/1400  ok=75 skip=0 fail=0
[18:46:26] progress 100/1400  ok=100 skip=0 fail=0
[18:46:36] progress 125/1400  ok=125 skip=0 fail=0
[18:46:50] progress 150/1400  ok=150 skip=0 fail=0
[18:47:00] progress 175/1400  ok=175 skip=0 fail=0
[18:47:13] progress 200/1400  ok=200 skip=0 fail=0
[18:47:23] progress 225/1400  ok=225 skip=0 fail=0
[18:47:34] progress 250/1400  ok=250 skip=0 fail=0
[18:47:45] progress 275/1400  ok=275 skip=0 fail=0
[18:47:56] progress 300/1400  ok=300 skip=0 fail=0
[18:48:06] progress 325/1400  ok=325 skip=0 fail=0
[18:48:17] progress 350/1400  ok=350 skip=0 fail=0
[18:48:28] progress 375/1400  ok=375 skip=0 fail=0
[18:48:39] progress 400/1400  ok=400 skip=0 fail=0
[18:48:49] progress 425

In [8]:
!python scripts/build_realworld_dataset.py

[18:57:20] converting WhatsApp voice notes...
/content/sih/ai_voice_detector/scripts/build_realworld_dataset.py:69: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(src, sr=16000, mono=True)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
[18:57:37] converted WhatsApp Audio 2026-09-01 at 17.48.07.mp4 -> whatsapp_WhatsApp_Audio_2026-09-01_at_17.48.07.wav
[18:57:37] converted WhatsApp Video 2026-09-01 at 5.34.43 PM.mp3.mpeg -> whatsapp_WhatsApp_Video_2026-09-01_at_5.34.43_PM.mp3.wav
[18:57:37] WhatsApp: 3 files converted
[18:57:37] downloading LibriSpeech dev-clean (https://www.openslr.org/resources/12/dev-clean.tar.gz)...
[18:59:37] download complete: 337926286 bytes
[18:59:37] scanning tarball for .flac entries...
[18:59:38] found 2703 

In [10]:
!pip install -q gtts requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.28.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [11]:
!python scripts/build_realworld_fake_dataset.py

[19:05:03] holdout exclude set: 413 files
[19:05:03] building modern TTS fake samples (gTTS)...
[19:05:12] loaded 200 candidate sentences from LibriSpeech transcripts
[19:05:24]   synthesized 25/200
[19:05:34]   synthesized 50/200
[19:05:44]   synthesized 75/200
[19:05:50]   synthesized 100/200
[19:06:01]   synthesized 125/200
[19:06:10]   synthesized 150/200
[19:06:17]   synthesized 175/200
[19:06:28]   synthesized 200/200
[19:06:28] modern TTS fake: 200/200 converted
[19:06:28] building degraded copies of ASVspoof fake samples...
[19:06:33]   degraded 50/200
[19:06:40]   degraded 100/200
[19:06:45]   degraded 150/200
[19:06:54]   degraded 200/200
[19:06:54] degraded ASVspoof copies: 200/200 converted
[19:06:54] DONE. data_realworld/fake/ now has 400 wav files (tts=200, degraded_asvspoof=200)


In [20]:
!cd /content/sih && git pull


Already up to date.


In [22]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f.lower().endswith('.zip'):
            print(os.path.join(root, f))

/content/drive/MyDrive/Notes.zip


## Indian dataset pipeline (steps 1-3, replayed here since they're cheap)

In [12]:
import os
os.environ['INDIEFAKE_ZIP_DIR'] = INDIEFAKE_ZIP_DIR
!python scripts/organize_indian_dataset.py

extracting drive-download-20260903T071406Z-1-001.zip ...
Traceback (most recent call last):
  File "/content/sih/ai_voice_detector/scripts/organize_indian_dataset.py", line 85, in <module>
    main()
    ~~~~^^
  File "/content/sih/ai_voice_detector/scripts/organize_indian_dataset.py", line 47, in main
    with zipfile.ZipFile(zip_path) as zf:
         ~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/usr/lib/python3.13/zipfile/__init__.py", line 1389, in __init__
    self.fp = io.open(file, filemode)
              ~~~~~~~^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/IndieFake  Dataset/drive-download-20260903T071406Z-1-001.zip'


In [ ]:
!python verify_indian_dataset.py

In [ ]:
!python scripts/degrade_indian_dataset.py

In [ ]:
# Idempotent -- these will report 0 new entries since the manifests are
# already committed to git and cloned above. Harmless to (re-)run.
!python scripts/build_holdout_manifest_indian.py
!python scripts/build_generator_manifest_indian.py

## Step 4: extract SSL embeddings on GPU

`features_ssl.py` now auto-detects CUDA and moves the model/inputs there. `MAX_WORKERS=1` in the script is intentional even here -- one process already saturates a single GPU; more workers would just fight over it via separate CUDA contexts.

In [ ]:
!python scripts/extract_indian_ssl_features.py

## Step 5: train the combined classifier

In [ ]:
!python train_ssl.py --indian

## Step 6: six-bucket evaluation (clean / real-world / Indian x real / fake)

In [ ]:
!python scripts/five_quadrant_eval.py

## Step 7: leave-one-generator-out, including the new 'indiefake' generator

In [ ]:
!python scripts/leave_one_generator_out_eval.py

## Bring results back

`models/*.joblib` and `features_indian.npy` are gitignored (large/regenerable) -- copy them to Drive to download. The new Indian embeddings in `cache/ssl_embeddings/` ARE meant to be committed (same convention as the existing ASVspoof/real-world cache) so any machine that later `git pull`s gets them for free -- this needs a GitHub token with push access to this repo, entered interactively below (never hardcode it in the notebook).

In [ ]:
!mkdir -p /content/drive/MyDrive/sih_indian_results
!cp models/voice_classifier_ssl_indian.joblib models/scaler_ssl_indian.joblib features_indian.npy /content/drive/MyDrive/sih_indian_results/
print('copied to Drive: sih_indian_results/')

In [ ]:
import getpass
token = getpass.getpass('GitHub personal access token (repo scope, push rights): ')
!git config user.email "jeevanarhack@gmail.com"
!git config user.name "Jeevan Sai V"
!git add cache/ssl_embeddings data_realworld/holdout_manifest.json data_realworld/generator_manifest.json
!git commit -m "Add Indian-accent SSL embeddings from Colab run"
!git push https://{token}@github.com/Jeevan-Cyber-Sai/sih.git HEAD:main